In [2]:
# Cell 1: import library

import pandas as pd
import numpy as np
from pathlib import Path
import warnings, time, json
import lightgbm as lgb 
import matplotlib.pyplot as plt 
import seaborn as sns

In [4]:
# Cell 2: load dataset and setup
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

bm = json.loads(Path("../data/processed/features_blocks_meta.json").read_text())
H, LAST_DAY, TARGET = bm["H"], bm["LAST_DAY"], bm["TARGET"]
NUM_B, CAT_B = bm["NUM"], bm["CAT"]
FEATS_B = NUM_B + CAT_B
BLOCKS = {int(k): v for k, v in bm["blocks"].items()}
TEST_START = LAST_DAY - H + 1

base = pd.read_parquet("../data/processed/features.parquet", columns=["d_num", "item_id", "cat_id", "sales"])
base["sales"] = base["sales"].astype("float32")
Y = base.pivot(index="item_id", columns="d_num", values="sales").reindex(columns=range(1, LAST_DAY + 1))
Y_np = Y.to_numpy()
pos_map = pd.Series(np.arange(len(Y)), index=Y.index.astype(str))
cat_arr = (base.drop_duplicates("item_id").set_index("item_id")["cat_id"]
           .reindex(Y.index).astype(str).to_numpy())

rows = []
for i in (1, 2, 3):
    val_end = TEST_START -1 -(3 - i) * H
    val_start = val_end - H + 1
    rows.append({"fold": i, "train_end": val_start -1, "val_start": val_start, "val_end": val_end})
folds = pd.DataFrame(rows)

def wape(y, p):
    m = ~np.isnan(y) & ~np.isnan(p)
    return float(np.abs(y[m] - p[m]).sum() / y[m].sum())

print("Y shape:", Y.shape, "| NaN cells:", int(np.isnan(Y_np).sum()))
print("total units:", int(np.nansum(Y_np)))
print("blocks:", BLOCKS, "| features:", len(NUM_B), "numeric +", len(CAT_B), "categorical")  # expect 26 + 7
for s in BLOCKS.values():
    assert (Path(f"../data/processed/features_s{s}.parquet").exists()), f"missing features_s{s}.parquet"
display(folds)

Y shape: (3049, 1941) | NaN cells: 1129842
total units: 7832248
blocks: {1: 7, 2: 14, 3: 21, 4: 28} | features: 26 numeric + 7 categorical


,fold,train_end,val_start,val_end
0,1,1829,1830,1857
1,2,1857,1858,1885
2,3,1885,1886,1913


In [9]:
# Cell 3: Train the P90, P50 and P10 block models

PARAMS_Q = dict(learning_rate=0.1, num_leaves=63, min_data_in_leaf=100,
                feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                seed=42, verbose=-1, force_row_wise=True)
ROUNDS = 200
QUANTILES = [0.9, 0.5, 0.1]

Yv = {int(f["fold"]): Y_np[:, int(f["val_start"]) - 1: int(f["val_end"])] for _, f in folds.iterrows()}

def run_quantile(alpha):
    feats = NUM_B + CAT_B
    cols = ["d_num", "sales"] + list(dict.fromkeys(["item_id"] + feats))
    P_run = {int(f["fold"]): np.full((len(Y), H), np.nan, dtype="float32") for _, f in folds.iterrows()}
    params = {**PARAMS_Q, "objective": "quantile", "alpha": alpha}

    for b, s in BLOCKS.items():
        tbl = pd.read_parquet(Path(f"../data/processed/features_s{s}.parquet"), columns=cols)
        for _, f in folds.iterrows():
            fo, te, vs = int(f["fold"]), int(f["train_end"]), int(f["val_start"])
            d0, d1 = vs + 7 * (b - 1), vs + 7 * b - 1
            assert d1 - s <= te, "features would reach beyond the cutoff"
            train = tbl[tbl["d_num"] <= te]
            val = tbl[(tbl["d_num"] >= d0) & (tbl["d_num"] <= d1)]
            dtrain = lgb.Dataset(train[feats], label=train["sales"],
                                 categorical_feature=CAT_B, free_raw_data=True)
            booster = lgb.train(params, dtrain, num_boost_round=ROUNDS)
            rows_i = pos_map[val["item_id"].astype(str)].to_numpy()
            P_run[fo][rows_i, val["d_num"].to_numpy() - vs] = booster.predict(val[feats])
            del dtrain, booster
        del tbl
    return P_run

PQ = {}
for alpha in QUANTILES:
    path = Path(f"../data/processed/pred_q{int(alpha * 100)}.npz")
    if path.exists():
        z = np.load(path)
        PQ[alpha] = {int(k.replace("fold", "")): z[k] for k in z.files}
        print(f"q{alpha}: loaded from disk")
        continue
    t0 = time.time()
    PQ[alpha] = run_quantile(alpha)
    np.savez_compressed(path, **{f"fold{fo}": arr for fo, arr in PQ[alpha].items()})
    print(f"q{alpha}: done in {(time.time() - t0) / 60:.1f} min | saved {path.name}")

for alpha in QUANTILES:
    miss = sum(int((~np.isnan(Yv[fo]) & np.isnan(PQ[alpha][fo])).sum()) for fo in Yv)
    allp = np.concatenate([PQ[alpha][fo][~np.isnan(PQ[alpha][fo])] for fo in Yv])
    print(f"q{alpha}: cells without a prediction: {miss} | share of predictions below 0.05: {(allp < 0.05).mean():.3f}")

q0.9: done in 6.8 min | saved pred_q90.npz
q0.5: done in 7.4 min | saved pred_q50.npz
q0.1: done in 7.9 min | saved pred_q10.npz
q0.9: cells without a prediction: 0 | share of predictions below 0.05: 0.031
q0.5: cells without a prediction: 0 | share of predictions below 0.05: 0.476
q0.1: cells without a prediction: 0 | share of predictions below 0.05: 0.922


In [13]:
# Cell 4: Coverage and pinball loss

def flat(Pd):
    m = [~np.isnan(Yv[fo]) for fo in Yv]
    y = np.concatenate([Yv[fo][k] for fo, k in zip(Yv, m)]).astype("float64")
    p = np.concatenate([Pd[fo][k] for fo, k in zip(Yv, m)]).astype("float64")
    return y, p

def pinball(y, p, a):
    d = y - p
    return float(np.mean(np.maximum(a * d, (a - 1) * d)))

def emp_q(Y_train, a, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        q = np.nanquantile(Y_train[:, -w:], a, axis=1)
    return np.repeat(np.nan_to_num(q, nan=0.0)[:, None], h, axis=1)

EQ = {a: {int(f["fold"]): emp_q(Y_np[:, : int(f["train_end"])], a) for _, f in folds.iterrows()}
      for a in QUANTILES}

rows = []
for a in sorted(QUANTILES):
    for name, Pd in (("LightGBM", PQ[a]), ("empirical 28d", EQ[a])):
        y, p = flat(Pd)
        lo, hi = float(np.mean(y < p)), float(np.mean(y <= p))
        rows.append({"quantile": a, "model": name, "pinball": pinball(y, p, a),
                     "P(y < pred)": lo, "P(y <= pred)": hi,
                     "target inside": bool(lo - 1e-9 <= a <= hi + 1e-9),
                     "total pred / total actual": float(p.sum() / y.sum())})
display(pd.DataFrame(rows).round(3))

print("\nP90 coverage per fold (LightGBM):")
for fo in Yv:
    k = ~np.isnan(Yv[fo])
    print(f"  fold {fo}: P(y < pred) = {np.mean(Yv[fo][k] < PQ[0.9][fo][k]):.3f} | "
          f"P(y <= pred) = {np.mean(Yv[fo][k] <= PQ[0.9][fo][k]):.3f}")

print("\nband [P10, P90] (target at least 0.80):")
for name, d_lo, d_hi in (("LightGBM", PQ[0.1], PQ[0.9]), ("empirical 28d", EQ[0.1], EQ[0.9])):
    y, lo = flat(d_lo)
    _, hi = flat(d_hi)
    print(f"  {name}: inside = {np.mean((y >= lo) & (y <= hi)):.3f} | mean width = {np.mean(hi - lo):.3f}")

_, p10 = flat(PQ[0.1]); _, p50 = flat(PQ[0.5]); _, p90 = flat(PQ[0.9])
print("\nquantile crossing: P10 > P50:", round(float(np.mean(p10 > p50 + 1e-9)), 4),
      "| P50 > P90:", round(float(np.mean(p50 > p90 + 1e-9)), 4),
      "| P10 > P90:", round(float(np.mean(p10 > p90 + 1e-9)), 4))

,quantile,model,pinball,P(y < pred),P(y <= pred),target inside,total pred / total actual
0,0.1,LightGBM,0.142,0.013,0.539,True,0.064
1,0.1,empirical 28d,0.144,0.016,0.557,True,0.186
2,0.5,LightGBM,0.496,0.315,0.621,True,0.715
3,0.5,empirical 28d,0.512,0.196,0.689,True,0.726
4,0.9,LightGBM,0.323,0.899,0.899,False,2.259
5,0.9,empirical 28d,0.354,0.699,0.906,True,2.050



P90 coverage per fold (LightGBM):
  fold 1: P(y < pred) = 0.899 | P(y <= pred) = 0.899
  fold 2: P(y < pred) = 0.899 | P(y <= pred) = 0.899
  fold 3: P(y < pred) = 0.898 | P(y <= pred) = 0.898

band [P10, P90] (target at least 0.80):
  LightGBM: inside = 0.886 | mean width = 3.224
  empirical 28d: inside = 0.890 | mean width = 2.737

quantile crossing: P10 > P50: 0.021 | P50 > P90: 0.0037 | P10 > P90: 0.0035


In [16]:
# Cell 5: Fix the crossing and save the sorted predictions
PQ_raw = PQ
PQ_sorted = {a: {} for a in QUANTILES}
for fo in Yv:
    s = np.sort(np.stack([PQ_raw[0.1][fo], PQ_raw[0.5][fo], PQ_raw[0.9][fo]]), axis=0)
    PQ_sorted[0.1][fo], PQ_sorted[0.5][fo], PQ_sorted[0.9][fo] = s[0], s[1], s[2]

for a in sorted(QUANTILES):
    y, p0 = flat(PQ_raw[a]); _, p1 = flat(PQ_sorted[a])
    print(f"q{a}: pinball before {pinball(y, p0, a):.4f} | after {pinball(y, p1, a):.4f}")

_, a10 = flat(PQ_sorted[0.1]); _, a50 = flat(PQ_sorted[0.5]); _, a90 = flat(PQ_sorted[0.9])
print("crossing after sorting:", float(np.mean(a10 > a50)), float(np.mean(a50 > a90)), float(np.mean(a10 > a90)))

for a in QUANTILES:
    np.savez_compressed(Path(f"../data/processed/pred_q{int(a * 100)}_sorted.npz"),
                        **{f"fold{fo}": arr for fo, arr in PQ_sorted[a].items()})
PQ = PQ_sorted
print("saved sorted predictions")

q0.1: pinball before 0.1425 | after 0.1425
q0.5: pinball before 0.4959 | after 0.4958
q0.9: pinball before 0.3231 | after 0.3231
crossing after sorting: 0.0 0.0 0.0
saved sorted predictions


In [17]:
# Cell 6: Is P90 calibrated in every subgroup?

base_date = pd.Timestamp("2011-01-29")                           # d_1
assert (base_date + pd.Timedelta(days=1829)).dayofweek == 0       # d_1830 = 2016-02-01, a Monday

recs = []
for _, f in folds.iterrows():
    fo, te, vs = int(f["fold"]), int(f["train_end"]), int(f["val_start"])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        vol28 = np.nanmean(Y_np[:, te - 28: te], axis=1)
    yy = Yv[fo]
    ii, dd = np.where(~np.isnan(yy))
    recs.append(pd.DataFrame({
        "fold": fo, "y": yy[ii, dd], "p90": PQ[0.9][fo][ii, dd],
        "cat": cat_arr[ii], "vol": np.nan_to_num(vol28[ii], nan=-1.0), "week": dd // 7 + 1,
        "dow": (base_date + pd.to_timedelta(vs - 1 + dd, unit="D")).dayofweek.to_numpy()}))
recs = pd.concat(recs, ignore_index=True)
assert len(recs) == sum(int((~np.isnan(Yv[fo])).sum()) for fo in Yv), "row count differs from valid cells"

recs["vol_bucket"] = pd.cut(recs["vol"], bins=[-1.5, -0.5, 0.1, 0.5, 1, 2, 5, 1e9],
                            labels=["no history", "<0.1", "0.1-0.5", "0.5-1", "1-2", "2-5", ">5"])
recs["lt"] = recs["y"] < recs["p90"]
recs["le"] = recs["y"] <= recs["p90"]

def table(by):
    return (recs.groupby(by, observed=True)
                .agg(cells=("y", "size"), strictly_below=("lt", "mean"), at_or_below=("le", "mean"),
                     mean_p90=("p90", "mean"), mean_actual=("y", "mean")).round(3))

for by in ("cat", "vol_bucket", "week", "dow", "fold"):
    print(f"\nP90 coverage by {by}:")
    display(table(by))


P90 coverage by cat:


,cells,strictly_below,at_or_below,mean_p90,mean_actual
cat,,,,,
FOODS,120696,0.898,0.899,4.406,2.059
HOBBIES,47460,0.899,0.905,2.538,0.928
HOUSEHOLD,87948,0.901,0.905,2.245,0.949



P90 coverage by vol_bucket:


,cells,strictly_below,at_or_below,mean_p90,mean_actual
vol_bucket,,,,,
no history,16,0.938,0.938,3.664000,0.625
<0.1,36680,0.902,0.924,0.868000,0.272
0.1-0.5,70308,0.897,0.897,1.243000,0.360
0.5-1,54768,0.897,0.897,2.209000,0.791
1-2,49280,0.900,0.900,3.420000,1.397
2-5,30912,0.900,0.900,6.376000,3.001
>5,14140,0.903,0.903,17.242001,9.609



P90 coverage by week:


,cells,strictly_below,at_or_below,mean_p90,mean_actual
week,,,,,
1,64022,0.908,0.909,3.328,1.495
2,64024,0.899,0.905,3.366,1.482
3,64029,0.899,0.902,3.304,1.444
4,64029,0.891,0.893,3.272,1.453



P90 coverage by dow:


,cells,strictly_below,at_or_below,mean_p90,mean_actual
dow,,,,,
0,36586,0.899,0.903,3.152,1.385
1,36586,0.901,0.905,2.968,1.235
2,36586,0.899,0.904,2.932,1.214
3,36586,0.899,0.903,2.949,1.236
4,36586,0.903,0.906,3.316,1.433
5,36587,0.897,0.898,3.969,1.882
6,36587,0.895,0.896,3.938,1.894



P90 coverage by fold:


,cells,strictly_below,at_or_below,mean_p90,mean_actual
fold,,,,,
1,85360,0.899,0.902,3.233,1.439
2,85372,0.899,0.903,3.335,1.467
3,85372,0.898,0.902,3.384,1.499


## Summary of Notebook 07: prediction intervals

**Setup**
- LightGBM quantile models (alpha 0.9, 0.5, 0.1), same design as Notebook 06: four horizon blocks, features shifted by 7/14/21/28 days, item_id, 63 leaves, learning rate 0.1, 200 rounds (not tuned for quantiles)
- Same 3 folds (2016-02-01 to 2016-04-24); final test window untouched; about 22 minutes for all quantiles
- Rival: empirical quantile of each item's last 28 days, repeated for 28 days
- Quantiles sorted per cell to remove crossing (before: P10>P50 in 2.1% of cells, P50>P90 in 0.37%; after: 0)

**Results (mean of 3 folds)**

| Quantile | Pinball LightGBM | Pinball empirical 28d | Coverage (strictly below / at or below) | Total predicted / total actual |
|---|---|---|---|---|
| P10 | 0.142 | 0.144 | 0.013 / 0.539 | 0.064 |
| P50 | 0.496 | 0.512 | 0.315 / 0.621 | 0.715 |
| P90 | 0.323 | 0.354 | 0.899 / 0.899 | 2.259 |

**Findings**
- P90 is calibrated: coverage 0.899 overall, 0.899/0.899/0.898 per fold, 0.898-0.901 by category, 0.900-0.903 for items selling 1+ units/day
- Small drifts: coverage 0.908 in horizon week 1 vs 0.891 in week 4; weekends 0.895-0.897 vs 0.899-0.903 on weekdays (not corrected; cells are correlated)
- LightGBM beats the empirical rival on pinball loss at all three quantiles; the gain is largest at P90 (8.8%), a tie at P10
- P10 and P50 coverage is not informative with small integer sales: most P10 predictions are 0 and the actual is 0 on over half the cells
- P50 under-forecasts totals by about 28% (as the L1 model did); P90 predictions sum to 2.26 times the units sold (ratio 1.8 for items over 5 units/day, 3.2 for items under 0.1 units/day)
- Band P10-P90 contains 0.886 of actuals (empirical rival 0.890) with mean width 3.22 vs 2.74; mostly P90 coverage, since sales are never below 0

**Limits (for the README)**
- Quantile settings reused from the Poisson model; rounds not tuned
- Calibration measured on the same three spring folds used elsewhere (no summer/holiday fold)
- Sorting is a post-fix; calibration was not adjusted
- Stock-level cost of P90 is measured in Notebook 09